In [3]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

In [7]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from src.scripts.train_stratified_rf import run_stratified_rf_training

In [8]:
bad_sessions = [ 21,  20, 162,  14, 163, 160, 167]

In [11]:
base_cols = [
            'target_queue_len', 'others_len_queue', 'iat_fqdn', 'num_running_funcs_filled',
            'gpu_warm_results_sec', 'gpu_cold_results_sec', 'is_cold_start'
        ]

In [9]:
def train_tune_and_print_results(path_to_training_data:str, base_cols=None):
    data = pd.read_csv(path_to_training_data)
    data = data[~data['session'].isin(bad_sessions)].reset_index(drop=True) ### FOCUS HERE: I'VE REMOVED SUSPICIOUS SESSIONS
    best_model, error_data = run_stratified_rf_training(data, base_cols, 'e2etime')
    return best_model, error_data

In [13]:
path_to_feature_data = "../data/processed/feature_data.csv"

In [14]:
rf_model, results_df = train_tune_and_print_results(path_to_feature_data, base_cols)

Original Dataset Size: 2680 rows

Policy Distribution Across Total Sessions:
dispatch_policy
Landlord    2
Name: count, dtype: int64

Stratified Split Complete!
Train Sessions: 1 | Test Sessions: 1

--- STARTING RANDOMIZED SEARCH CV TUNING ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Optimal Hyperparameters Discovered:
 - n_estimators: 100
 - min_samples_leaf: 5
 - max_features: log2
 - max_depth: 15

--- PERFORMANCE METRICS ---
Tuned Train R2 (Log): 0.8279
Tuned Test R2  (Log): 0.7767
Tuned Test MSE (Real): 2.9006
Tuned Test MAE (Real): 0.5037

--- FEATURE IMPORTANCES ---
                 Feature  Importance
    gpu_warm_results_sec    0.273120
    gpu_cold_results_sec    0.229764
           is_cold_start    0.175509
                iat_fqdn    0.173619
        others_len_queue    0.089410
num_running_funcs_filled    0.058577
        target_queue_len    0.000000


In [15]:
rf_model.feature_names_in_

array(['target_queue_len', 'others_len_queue', 'iat_fqdn',
       'num_running_funcs_filled', 'gpu_warm_results_sec',
       'gpu_cold_results_sec', 'is_cold_start'], dtype=object)

In [ ]:
import joblib

# Save tuned RF model to a file
# Use .joblib  as the extension
joblib.dump(rf_model, '../models/tuned_faas_rf_proportionate_data_7_features_final.joblib')

print("Model saved successfully!")

In [17]:
results_df.to_csv('../data/processed/results/error_data/rf_model_results.csv', index=False)